# Batch 1 — Training (Baseline)

Trains and tunes 9 models on the fixed training split from `processing.ipynb`,
using 5-fold stratified cross-validation and **PR-AUC (average precision)**
as the primary selection metric — not accuracy, since the target is ~10.7%
fraud / ~89.3% not fraud, and a model that always predicts "not fraud"
would already score ~89% accuracy while being useless.

Two deliberate exceptions to 5-fold CV, documented where they occur:
- **SVM** is tuned on a stratified subsample of the training set (RBF-kernel
  SVC scales roughly O(n²)–O(n³), which is impractical on the full ~35,000
  training rows).
- **MLP** uses 3-fold CV instead of 5 to keep total runtime reasonable,
  since each fit is noticeably more expensive than the tree-based models.

No class weighting or resampling is applied anywhere in this batch —
imbalance handling is Batch 2's job. This batch measures the baseline.


In [1]:
import time
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import make_scorer, average_precision_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
RANDOM_STATE = 42


## 1. Paths & Load Artifacts

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'Models_Batch_1' else Path.cwd()
BATCH_DIR = PROJECT_ROOT / 'Models_Batch_1'
ARTIFACTS_DIR = BATCH_DIR / 'artifacts'
MODELS_DIR = ARTIFACTS_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

data = joblib.load(ARTIFACTS_DIR / 'batch1_data.joblib')

y_train, y_val = data['y_train'], data['y_val']
cat_feature_indices = data['cat_feature_indices']

print("Loaded batch1_data.joblib")
print(f"Train rows: {len(y_train)}, fraud rate: {y_train.mean():.4f}")
print(f"Val rows:   {len(y_val)}, fraud rate: {y_val.mean():.4f}")


Loaded batch1_data.joblib
Train rows: 35000, fraud rate: 0.1072
Val rows:   7500, fraud rate: 0.1072


## 2. CV Strategy & Shared Search Helper

`average_precision` (PR-AUC) is the metric the search optimizes for and
selects the best hyperparameters by (`refit='pr_auc'`). We also score every
CV candidate on recall, precision, F1, and ROC-AUC in the same search pass
(via a multi-metric `scoring` dict) so we get every number we need without
re-running cross-validation separately.


In [3]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

SCORING = {
    'pr_auc': 'average_precision',
    'recall': 'recall',
    'precision': 'precision',
    'f1': 'f1',
    'roc_auc': 'roc_auc',
}

results = []  # collects one dict per model for the comparison table

def run_experiment(name, estimator, param_distributions, X_train, X_val,
                    search_type='random', n_iter=20, cv=cv5, search_n_jobs=1,
                    preprocessing='Encoded (unscaled)', feature_set='tree',
                    sampling='None', notes='', fit_params=None):
    """Fit a hyperparameter search, evaluate the best estimator on validation,
    save the model, and append one row to the shared `results` list."""
    print(f"\n{'='*70}\n{name}\n{'='*70}")
    t0 = time.time()

    if search_type == 'random':
        search = RandomizedSearchCV(
            estimator, param_distributions, n_iter=n_iter, cv=cv,
            scoring=SCORING, refit='pr_auc', random_state=RANDOM_STATE,
            n_jobs=search_n_jobs, verbose=0,
        )
    else:
        search = GridSearchCV(
            estimator, param_distributions, cv=cv,
            scoring=SCORING, refit='pr_auc', n_jobs=search_n_jobs, verbose=0,
        )

    search.fit(X_train, y_train, **(fit_params or {}))
    elapsed = time.time() - t0

    best_idx = search.best_index_
    cv_pr_auc = search.cv_results_['mean_test_pr_auc'][best_idx]
    cv_pr_auc_std = search.cv_results_['std_test_pr_auc'][best_idx]
    cv_recall = search.cv_results_['mean_test_recall'][best_idx]
    cv_precision = search.cv_results_['mean_test_precision'][best_idx]
    cv_f1 = search.cv_results_['mean_test_f1'][best_idx]
    cv_roc_auc = search.cv_results_['mean_test_roc_auc'][best_idx]

    best_model = search.best_estimator_
    val_proba = best_model.predict_proba(X_val)[:, 1]
    val_pred = (val_proba >= 0.5).astype(int)

    from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score
    val_pr_auc = average_precision_score(y_val, val_proba)
    val_recall = recall_score(y_val, val_pred)
    val_precision = precision_score(y_val, val_pred, zero_division=0)
    val_f1 = f1_score(y_val, val_pred)
    val_roc_auc = roc_auc_score(y_val, val_proba)

    print(f"Best params: {search.best_params_}")
    print(f"CV  -> PR-AUC: {cv_pr_auc:.4f} (+/- {cv_pr_auc_std:.4f})  Recall: {cv_recall:.4f}  Precision: {cv_precision:.4f}  F1: {cv_f1:.4f}  ROC-AUC: {cv_roc_auc:.4f}")
    print(f"Val -> PR-AUC: {val_pr_auc:.4f}  Recall: {val_recall:.4f}  Precision: {val_precision:.4f}  F1: {val_f1:.4f}  ROC-AUC: {val_roc_auc:.4f}")
    print(f"Elapsed: {elapsed:.1f}s")

    joblib.dump(best_model, MODELS_DIR / f'{name}.joblib')

    results.append({
        'Batch': 'Batch_1', 'Model': name, 'Preprocessing': preprocessing,
        'Feature_Set': feature_set, 'Sampling_Technique': sampling,
        'Best_Parameters': str(search.best_params_),
        'CV_PR_AUC': round(cv_pr_auc, 4), 'CV_PR_AUC_std': round(cv_pr_auc_std, 4),
        'Validation_PR_AUC': round(val_pr_auc, 4),
        'Precision': round(val_precision, 4), 'Recall': round(val_recall, 4),
        'F1': round(val_f1, 4), 'ROC_AUC': round(val_roc_auc, 4),
        'CV_Recall': round(cv_recall, 4), 'CV_Precision': round(cv_precision, 4),
        'CV_F1': round(cv_f1, 4), 'CV_ROC_AUC': round(cv_roc_auc, 4),
        'Train_Time_s': round(elapsed, 1), 'Notes': notes,
    })
    return search


## 3. Logistic Regression

Scale-sensitive — tuned separately on the `StandardScaler` and
`RobustScaler` feature sets, keeping whichever gives the higher CV PR-AUC.
`solver='liblinear'` supports both L1 and L2 penalties and is efficient at
this dataset size.


In [4]:
logreg_params = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],
}

search_logreg_std = run_experiment(
    'LogisticRegression_std', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    logreg_params, data['X_train_std'], data['X_val_std'],
    search_type='grid', search_n_jobs=-1,
    preprocessing='StandardScaler', feature_set='std', notes='Baseline, no class weighting'
)

search_logreg_rob = run_experiment(
    'LogisticRegression_rob', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    logreg_params, data['X_train_rob'], data['X_val_rob'],
    search_type='grid', search_n_jobs=-1,
    preprocessing='RobustScaler', feature_set='rob', notes='Baseline, no class weighting'
)

# Keep whichever scaler won by CV PR-AUC as "the" Logistic Regression result
best_scaler_logreg = 'std' if results[-2]['CV_PR_AUC'] >= results[-1]['CV_PR_AUC'] else 'rob'
print(f"\nBetter scaler for Logistic Regression: {best_scaler_logreg}")



LogisticRegression_std


Best params: {'C': 1, 'penalty': 'l1', 'solver': 'liblinear'}
CV  -> PR-AUC: 0.7242 (+/- 0.0134)  Recall: 0.4532  Precision: 0.8875  F1: 0.5999  ROC-AUC: 0.8873
Val -> PR-AUC: 0.7076  Recall: 0.4403  Precision: 0.8784  F1: 0.5866  ROC-AUC: 0.8883
Elapsed: 33.6s

LogisticRegression_rob


Best params: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}
CV  -> PR-AUC: 0.7255 (+/- 0.0131)  Recall: 0.4452  Precision: 0.8894  F1: 0.5933  ROC-AUC: 0.8879
Val -> PR-AUC: 0.7087  Recall: 0.4366  Precision: 0.8775  F1: 0.5831  ROC-AUC: 0.8889
Elapsed: 29.7s

Better scaler for Logistic Regression: rob


## 4. Decision Tree

Unscaled encoded features — trees split on raw thresholds, scaling has no effect.

In [5]:
dt_params = {
    'max_depth': [3, 5, 7, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10],
    'criterion': ['gini', 'entropy'],
}

search_dt = run_experiment(
    'DecisionTree', DecisionTreeClassifier(random_state=RANDOM_STATE),
    dt_params, data['X_train_tree'], data['X_val_tree'],
    search_type='random', n_iter=25, search_n_jobs=-1,
    preprocessing='None (unscaled)', feature_set='tree', notes='Baseline, no class weighting'
)



DecisionTree


Best params: {'min_samples_split': 20, 'min_samples_leaf': 1, 'max_depth': 7, 'criterion': 'entropy'}
CV  -> PR-AUC: 0.7475 (+/- 0.0163)  Recall: 0.5867  Precision: 0.8748  F1: 0.7023  ROC-AUC: 0.8947
Val -> PR-AUC: 0.7585  Recall: 0.5759  Precision: 0.9260  F1: 0.7101  ROC-AUC: 0.9004
Elapsed: 26.0s


## 5. Random Forest

In [6]:
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
}

search_rf = run_experiment(
    'RandomForest', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    rf_params, data['X_train_tree'], data['X_val_tree'],
    search_type='random', n_iter=15, search_n_jobs=1,
    preprocessing='None (unscaled)', feature_set='tree', notes='Baseline, no class weighting'
)



RandomForest


C:\Users\Ali Ahmed\AppData\Local\Programs\Python\Python38\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Best params: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': None}
CV  -> PR-AUC: 0.7809 (+/- 0.0125)  Recall: 0.5332  Precision: 0.9413  F1: 0.6806  ROC-AUC: 0.9081
Val -> PR-AUC: 0.7724  Recall: 0.5137  Precision: 0.9302  F1: 0.6619  ROC-AUC: 0.9140
Elapsed: 426.5s


## 6. XGBoost

In [7]:
xgb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
}

search_xgb = run_experiment(
    'XGBoost', XGBClassifier(eval_metric='logloss', tree_method='hist',
                              random_state=RANDOM_STATE, n_jobs=-1),
    xgb_params, data['X_train_tree'], data['X_val_tree'],
    search_type='random', n_iter=15, search_n_jobs=1,
    preprocessing='None (unscaled)', feature_set='tree', notes='Baseline, no class weighting'
)



XGBoost


Best params: {'subsample': 0.7, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 0.7}
CV  -> PR-AUC: 0.8037 (+/- 0.0147)  Recall: 0.6264  Precision: 0.8929  F1: 0.7362  ROC-AUC: 0.9222
Val -> PR-AUC: 0.8039  Recall: 0.6132  Precision: 0.9079  F1: 0.7320  ROC-AUC: 0.9293
Elapsed: 54.9s


## 7. LightGBM

In [8]:
lgbm_params = {
    'n_estimators': [100, 200, 300],
    'num_leaves': [15, 31, 63, 127],
    'max_depth': [-1, 5, 10, 15],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
}

search_lgbm = run_experiment(
    'LightGBM', LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1),
    lgbm_params, data['X_train_tree'], data['X_val_tree'],
    search_type='random', n_iter=15, search_n_jobs=1,
    preprocessing='None (unscaled)', feature_set='tree', notes='Baseline, no class weighting'
)



LightGBM


Best params: {'subsample': 0.7, 'num_leaves': 63, 'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 0.7}
CV  -> PR-AUC: 0.8006 (+/- 0.0140)  Recall: 0.6264  Precision: 0.8898  F1: 0.7351  ROC-AUC: 0.9199
Val -> PR-AUC: 0.8008  Recall: 0.6144  Precision: 0.9015  F1: 0.7308  ROC-AUC: 0.9291
Elapsed: 68.8s


## 8. CatBoost

Uses the **native categorical** feature set (`Business_Type`, `Region` as
raw categories) instead of one-hot encoding, since CatBoost handles
categoricals internally via target statistics — this is the "investigate
using categorical features natively" preprocessing note from the project
plan.

Note: `cat_features` is passed to **`.fit()`**, not the constructor.
CatBoost's sklearn wrapper doesn't round-trip `cat_features` correctly
through `get_params()`/`clone()`, which makes `RandomizedSearchCV` raise a
`RuntimeError` if it's set at construction time — passing it as a fit
parameter avoids that entirely.


In [9]:
catboost_params = {
    'iterations': [200, 400, 600],
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'l2_leaf_reg': [1, 3, 5, 7],
}

search_catboost = run_experiment(
    'CatBoost',
    CatBoostClassifier(random_state=RANDOM_STATE, verbose=0, thread_count=-1),
    catboost_params, data['X_train_cb'], data['X_val_cb'],
    search_type='random', n_iter=10, search_n_jobs=1,
    preprocessing='Native categorical (no one-hot)', feature_set='catboost_native',
    notes='Baseline, no class weighting',
    fit_params={'cat_features': cat_feature_indices},
)



CatBoost


Best params: {'learning_rate': 0.1, 'l2_leaf_reg': 7, 'iterations': 200, 'depth': 4}
CV  -> PR-AUC: 0.8041 (+/- 0.0150)  Recall: 0.6171  Precision: 0.9062  F1: 0.7341  ROC-AUC: 0.9222
Val -> PR-AUC: 0.8051  Recall: 0.5995  Precision: 0.9181  F1: 0.7254  ROC-AUC: 0.9314
Elapsed: 1518.7s


## 9. SVM

RBF-kernel `SVC` scales roughly O(n²)–O(n³) with training set size, which
is impractical to tune (or even fit once per CV fold) on the full ~35,000
training rows within a reasonable experiment budget. We tune **and** fit
the final model on a **stratified subsample of 8,000 training rows**
(same fraud rate preserved) — a documented, deliberate limitation, not an
oversight. StandardScaler features are used (the better scaler is picked
from the Logistic Regression comparison as a reasonable default for this
subsample too).


In [10]:
SVM_SAMPLE_SIZE = 6000
svm_scaler_key = f'X_train_{best_scaler_logreg}'
X_train_svm_full = data[svm_scaler_key]

X_svm_sample, _, y_svm_sample, _ = train_test_split(
    X_train_svm_full, y_train, train_size=SVM_SAMPLE_SIZE,
    stratify=y_train, random_state=RANDOM_STATE
)
print(f"SVM training subsample: {len(y_svm_sample)} rows, fraud rate = {y_svm_sample.mean():.4f}")


SVM training subsample: 6000 rows, fraud rate = 0.1072


In [11]:
svm_params = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
    'kernel': ['rbf'],
}

# temporarily point y_train at the SVM subsample labels for this one search
# (run_experiment reads the module-level y_train for CV scoring)
_y_train_backup = y_train
y_train = y_svm_sample

search_svm = run_experiment(
    'SVM', SVC(probability=True, random_state=RANDOM_STATE, max_iter=20000),
    svm_params, X_svm_sample, data[f'X_val_{best_scaler_logreg}'],
    search_type='random', n_iter=8, cv=cv3, search_n_jobs=-1,
    preprocessing=f'{"StandardScaler" if best_scaler_logreg=="std" else "RobustScaler"} (8k subsample)',
    feature_set=best_scaler_logreg,
    notes=f'Trained on stratified {SVM_SAMPLE_SIZE}-row subsample of train (RBF SVC does not scale to full 35k rows)'
)

y_train = _y_train_backup  # restore full-size labels for remaining models



SVM


Best params: {'kernel': 'rbf', 'gamma': 'auto', 'C': 10}
CV  -> PR-AUC: 0.7708 (+/- 0.0211)  Recall: 0.5754  Precision: 0.8599  F1: 0.6894  ROC-AUC: 0.8997
Val -> PR-AUC: 0.7288  Recall: 0.5460  Precision: 0.8642  F1: 0.6692  ROC-AUC: 0.8841
Elapsed: 18.0s


## 10. Naive Bayes

`GaussianNB` is scale-invariant, so it uses the unscaled encoded feature set. Effectively only `var_smoothing` is tunable.

In [12]:
nb_params = {
    'var_smoothing': np.logspace(-11, -7, 9),
}

search_nb = run_experiment(
    'NaiveBayes', GaussianNB(),
    nb_params, data['X_train_tree'], data['X_val_tree'],
    search_type='grid', search_n_jobs=-1,
    preprocessing='None (unscaled)', feature_set='tree', notes='Baseline, no class weighting'
)



NaiveBayes


Best params: {'var_smoothing': 1e-11}
CV  -> PR-AUC: 0.4342 (+/- 0.0179)  Recall: 0.2784  Precision: 0.6194  F1: 0.3841  ROC-AUC: 0.7644
Val -> PR-AUC: 0.4419  Recall: 0.2799  Precision: 0.6233  F1: 0.3863  ROC-AUC: 0.7708
Elapsed: 2.2s


## 11. Neural Network (MLP)

Scale-sensitive — uses the better scaler found for Logistic Regression.
3-fold CV (instead of 5) to keep total runtime reasonable, since MLP fits
are more expensive than the tree-based models.


In [13]:
mlp_params = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50)],
    'alpha': [0.0001, 0.001, 0.01, 0.1],
    'learning_rate_init': [0.001, 0.01],
    'activation': ['relu', 'tanh'],
}

search_mlp = run_experiment(
    'MLP', MLPClassifier(max_iter=200, early_stopping=True, random_state=RANDOM_STATE),
    mlp_params, data[f'X_train_{best_scaler_logreg}'], data[f'X_val_{best_scaler_logreg}'],
    search_type='random', n_iter=8, cv=cv3, search_n_jobs=-1,
    preprocessing=f'{"StandardScaler" if best_scaler_logreg=="std" else "RobustScaler"}',
    feature_set=best_scaler_logreg, notes='Baseline, no class weighting'
)



MLP


Best params: {'learning_rate_init': 0.001, 'hidden_layer_sizes': (50, 50), 'alpha': 0.01, 'activation': 'tanh'}
CV  -> PR-AUC: 0.7891 (+/- 0.0116)  Recall: 0.6086  Precision: 0.8999  F1: 0.7261  ROC-AUC: 0.9057
Val -> PR-AUC: 0.7804  Recall: 0.6032  Precision: 0.8932  F1: 0.7201  ROC-AUC: 0.9079
Elapsed: 22.5s


## 12. Save Batch 1 Results Table

In [14]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Validation_PR_AUC', ascending=False).reset_index(drop=True)

results_df.to_csv(ARTIFACTS_DIR / 'batch1_results.csv', index=False)
print(f"Saved {len(results_df)} model results to batch1_results.csv")
results_df[['Model', 'Feature_Set', 'CV_PR_AUC', 'Validation_PR_AUC', 'Precision', 'Recall', 'F1', 'ROC_AUC']]


Saved 10 model results to batch1_results.csv


,Model,Feature_Set,CV_PR_AUC,Validation_PR_AUC,Precision,Recall,F1,ROC_AUC
0,CatBoost,catboost_native,0.8041,0.8051,0.9181,0.5995,0.7254,0.9314
1,XGBoost,tree,0.8037,0.8039,0.9079,0.6132,0.7320,0.9293
2,LightGBM,tree,0.8006,0.8008,0.9015,0.6144,0.7308,0.9291
3,MLP,rob,0.7891,0.7804,0.8932,0.6032,0.7201,0.9079
4,RandomForest,tree,0.7809,0.7724,0.9302,0.5137,0.6619,0.9140
5,DecisionTree,tree,0.7475,0.7585,0.9260,0.5759,0.7101,0.9004
6,SVM,rob,0.7708,0.7288,0.8642,0.5460,0.6692,0.8841
7,LogisticRegression_rob,rob,0.7255,0.7087,0.8775,0.4366,0.5831,0.8889
8,LogisticRegression_std,std,0.7242,0.7076,0.8784,0.4403,0.5866,0.8883
9,NaiveBayes,tree,0.4342,0.4419,0.6233,0.2799,0.3863,0.7708


---
**Next:** `evaluation.ipynb` loads these results and saved models, produces
confusion matrices / ROC / PR curves, checks for overfitting (CV vs
validation gap) and CV stability, and writes up the Batch 1 analysis that
will drive the design of Batch 2.
